In [ ]:
# REGERENCES : 

# https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0188629 
 
#   https://ceur-ws.org/Vol-2485/paper32.pdf

# https://www.youtube.com/playlist?list=PLtGXgNsNHqPTgP9wyR8pmy2EuM2ZGHU5Z

In [2]:
 from glob import glob
import os 
import mne 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [3]:
edfs_folder = "edfs"

edfs_files = glob(os.path.join(edfs_folder,"*.edf"))

In [4]:
healthy_files = []
patient_files = []

for i in edfs_files:
    _ , file = i.split("/")

    if 's' in file:
        patient_files.append(os.path.join(edfs_folder,file))
    if 'h' in file:
        healthy_files.append(os.path.join(edfs_folder,file))


In [5]:
def read_data(file_path):
    data = mne.io.read_raw_edf(file_path,preload=True)
    data.set_eeg_reference()
    data.filter(l_freq=0.5,h_freq=45)
    epochs = mne.make_fixed_length_epochs(data,duration=5,overlap=1)
    array = epochs.get_data()
    return array

In [6]:
sample = read_data(healthy_files[0])

Extracting EDF parameters from /Users/lazycodebaker/Documents/codes/practises/AI/projects/EGG/schizophernia/edfs/h01.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 231249  =      0.000 ...   924.996 secs...
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 1651 samples (6.604 s)

Not setting metadata
231 matching events found
No baseline co

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


In [7]:
sample.shape

(231, 19, 1250)

In [8]:
%%capture

healthy_data = [read_data(h) for h in healthy_files]
patient_data = [read_data(p) for p in patient_files]

In [9]:
healthy_data[0].shape , healthy_data[1].shape

((231, 19, 1250), (216, 19, 1250))

In [10]:
healthy_labels = [ len(h_d)*[0] for h_d in healthy_data ]
patient_labels = [ len(p_d) * [0] for p_d in patient_data]

In [11]:
len(healthy_labels) , len(patient_labels)

(14, 14)

In [12]:
data_list = healthy_data + patient_data
labels_list = healthy_labels + patient_labels

In [13]:
group_list=[[i]*len(j) for i,j in enumerate(data_list)]

In [14]:
data_arr = np.vstack(data_list)
label_arr = np.hstack(labels_list)
group_arr = np.hstack(group_list)

print(data_arr.shape , label_arr.shape , group_arr.shape)

(7201, 19, 1250) (7201,) (7201,)


In [15]:
np.mean(data_arr,axis=-1).shape, np.mean(data_arr,axis=1).shape

((7201, 19), (7201, 1250))

In [16]:
from scipy import stats


In [18]:
mean_x = lambda x : np.mean(x,axis=-1)
std_x = lambda x : np.std(x,axis=-1)
var_x = lambda x : np.var(x,axis=-1)
ptp_x = lambda x : np.ptp(x,axis=-1)
minim = lambda x : np.min(x,axis=-1)
maxim = lambda x : np.max(x,axis=-1)
argmin = lambda x : np.argmin(x,axis=-1)
argmax= lambda x : np.argmax(x,axis=-1)
rms = lambda x : np.sqrt(np.mean(x**2 ,axis=-1))
abs_diff_signal = lambda x : np.sum(np.abs(np.diff(x,axis=-1)),axis=-1)
skewness = lambda x : stats.skew(x,axis=-1)
kurtosis = lambda x : stats.kurtosis(x,axis=-1)

concatenate_features = lambda x : np.concatenate((mean_x(x),std_x(x),ptp_x(x),var_x(x),
                                                  minim(x),maxim(x),argmin(x),argmax(x),rms(x),abs_diff_signal(x),skewness(x),kurtosis(x)),axis=-1)

In [19]:
features = []

for _d in data_arr:
    features.append(concatenate_features(_d))

features_arr = np.array(features)

In [22]:
features_arr.shape

(7201, 228)

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV , GroupKFold

In [23]:
clf = LogisticRegression() 
gfk =  GroupKFold(5)

pipeline = Pipeline([
    ('scaler',StandardScaler()),
    ('clf',clf)
])

param_grid = {
    'clf__penalty' : ['l1','l2'],
    'clf__C' : [0.1,0.5,0.7,1,3,5,7]
}

gscv = GridSearchCV(pipeline,param_grid=param_grid,scoring='accuracy',cv=gfk,n_jobs=12)
gscv.fit(features_arr,label_arr,groups=group_arr)

ValueError: 
All the 70 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
35 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/lazycodebaker/Documents/codes/practises/AI/projects/.conda/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/lazycodebaker/Documents/codes/practises/AI/projects/.conda/lib/python3.11/site-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lazycodebaker/Documents/codes/practises/AI/projects/.conda/lib/python3.11/site-packages/sklearn/pipeline.py", line 473, in fit
    self._final_estimator.fit(Xt, y, **l

In [24]:
data_arr = np.vstack(data_list)
label_arr = np.hstack(labels_list)
group_arr = np.hstack(group_list)

print(data_arr.shape , label_arr.shape , group_arr.shape)

(7201, 19, 1250) (7201,) (7201,)


In [25]:
data_arr = np.moveaxis(data_arr,1,2)
data_arr.shape

(7201, 1250, 19)

In [29]:
from tensorflow.keras.layers import Conv1D , BatchNormalization , LeakyReLU , MaxPool1D , GlobalAveragePooling1D , Dense, Dropout ,AveragePooling1D

In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.backend import clear_session

def cnn_model():
    clear_session()
    model=Sequential()
    model.add(Conv1D(filters=5,kernel_size=3,strides=1,input_shape=(6250,19)))#1
    model.add(BatchNormalization())
    model.add(LeakyReLU())
    model.add(MaxPool1D(pool_size=2,strides=2))#2
    model.add(Conv1D(filters=5,kernel_size=3,strides=1))#3
    model.add(LeakyReLU())
    model.add(MaxPool1D(pool_size=2,strides=2))#4
    model.add(Dropout(0.5))
    model.add(Conv1D(filters=5,kernel_size=3,strides=1))#5
    model.add(LeakyReLU())
    model.add(AveragePooling1D(pool_size=2,strides=2))#6
    model.add(Dropout(0.5))
    model.add(Conv1D(filters=5,kernel_size=3,strides=1))#7
    model.add(LeakyReLU())
    model.add(AveragePooling1D(pool_size=2,strides=2))#8
    model.add(Conv1D(filters=5,kernel_size=3,strides=1))#9
    model.add(LeakyReLU())
    model.add(GlobalAveragePooling1D())#10
    model.add(Dense(1,activation='sigmoid'))#11
    
    model.compile('adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model


model = cnn_model()

model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 6248, 5)        │           290 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 6248, 5)        │            20 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 6248, 5)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 3124, 5)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 3122, 5)        │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 3122, 5)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 1561, 5)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1561, 5)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 1559, 5)        │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 1559, 5)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling1d               │ (None, 779, 5)         │             0 │
│ (AveragePooling1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 779, 5)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 777, 5)         │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 777, 5)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling1d_1             │ (None, 388, 5)         │             0 │
│ (AveragePooling1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 386, 5)         │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 386, 5)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 5)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             6 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 636 (2.48 KB)

 Trainable params: 626 (2.45 KB)

 Non-trainable params: 10 (40.00 B)

In [32]:
from sklearn.model_selection import GroupKFold,LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
gkf=GroupKFold()

In [33]:
accuracy=[]

for train_index, val_index in gkf.split(data_arr, label_arr, groups=group_arr):
    train_features,train_labels=data_arr[train_index],label_arr[train_index]
    val_features,val_labels=data_arr[val_index],label_arr[val_index]
    scaler=StandardScaler()
    train_features = scaler.fit_transform(train_features.reshape(-1, train_features.shape[-1])).reshape(train_features.shape)
    val_features = scaler.transform(val_features.reshape(-1, val_features.shape[-1])).reshape(val_features.shape)
    model=cnn_model()
    model.fit(train_features,train_labels,epochs=50,batch_size=128,validation_data=(val_features,val_labels))
    accuracy.append(model.evaluate(val_features,val_labels)[1])

/Users/lazycodebaker/Documents/codes/practises/AI/projects/.conda/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 54ms/step - accuracy: 0.0793 - loss: 0.8189 - val_accuracy: 0.9931 - val_loss: 0.6431
Epoch 2/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.9958 - loss: 0.6240 - val_accuracy: 1.0000 - val_loss: 0.4664
Epoch 3/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 1.0000 - loss: 0.3390 - val_accuracy: 1.0000 - val_loss: 0.0476
Epoch 4/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 1.0000 - loss: 0.0177 - val_accuracy: 1.0000 - val_loss: 0.0079
Epoch 5/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 1.0000 - loss: 0.0039 - val_accuracy: 1.0000 - val_loss: 0.0039
Epoch 6/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 1.0000 - loss: 0.0020 - val_accuracy: 1.0000 - val_loss: 0.0025
Epoch 7/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 1.0000 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 0.0017
Epoch 8/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 83ms/step - accuracy: 1.0000 - loss: 6.4708e-04 - val_accuracy: 1.0000

/Users/lazycodebaker/Documents/codes/practises/AI/projects/.conda/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - accuracy: 0.9251 - loss: 0.6194 - val_accuracy: 1.0000 - val_loss: 0.3159
Epoch 2/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 72ms/step - accuracy: 1.0000 - loss: 0.1662 - val_accuracy: 1.0000 - val_loss: 0.0106
Epoch 3/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 1.0000 - loss: 0.0053 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 4/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 1.0000 - loss: 0.0022 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 5/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 6.2407e-04
Epoch 6/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 3.9070e-04
Epoch 7/50
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 1.0000 - loss: 5.6604e-04 - val_accuracy: 1.0000 - val_loss: 2.6590e-04
Epoch 8/50
27/45 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - accuracy: 1.0000 - loss: 2.6252e-04

KeyboardInterrupt: 

In [ ]:
# paper -2 https://www.youtube.com/watch?v=pmy3x1xNAF8&list=PLtGXgNsNHqPTgP9wyR8pmy2EuM2ZGHU5Z&index=4

In [34]:
!pip3 install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 MB 6.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 3.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 528.6 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 1.1 MB/s eta 0:00:00a 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 1.6 MB/s eta 0:00:00-:--:--


In [ ]:
import torch.nn as nn
import torch

https://www.kaggle.com/code/owaiskhan9654/training-of-eeg-schizophrenia-disorder-using-cnn



In [ ]:
https://www.youtube.com/watch?v=_BdBJOOqMes&list=PLtGXgNsNHqPTgP9wyR8pmy2EuM2ZGHU5Z&index=1 